In [ ]:
!pip install -r requirements.txt

In [ ]:
import base64
import json
from datetime import datetime

import numpy as np


def cosine_sim(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    magnitude_vector1 = np.linalg.norm(vector1)
    magnitude_vector2 = np.linalg.norm(vector2)

    if magnitude_vector1 == 0 or magnitude_vector2 == 0:
        return 0  # Handle cases where one or both vectors are zero vectors

    cosine_similarity = dot_product / (magnitude_vector1 * magnitude_vector2)
    return cosine_similarity


def load_file_as_base64(file_path):
    with open(file_path, "rb") as file:
        return base64.b64encode(file.read()).decode("utf-8")


def convert_datetime(obj):
    if isinstance(obj, dict):
        return {k: convert_datetime(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_datetime(item) for item in obj]
    elif isinstance(obj, datetime):
        return obj.isoformat()
    return obj


def pretty_format(obj, indent=2):
    obj_json_safe = convert_datetime(obj)
    return json.dumps(obj_json_safe, indent=indent)


# S3 Bucket Setup 

Create a unique S3 bucket and store it in a config for use in other notebook

In [ ]:
import boto3
import json

# Get AWS account ID
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
print(f"AWS Account ID: {account_id}")

# Create unique S3 bucket name
s3_bucket = f"nova-mme-get-started-{account_id}"
print(f"S3 Bucket Name: {s3_bucket}")

In [ ]:
# Store bucket name in a config file for other notebooks
config = {
    "s3_bucket": s3_bucket,
    "account_id": account_id
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Configuration saved to config.json")
print(f"S3 bucket name: {s3_bucket}")

In [ ]:
# Create S3 bucket
s3 = boto3.client('s3')

try:
    s3.create_bucket(Bucket=s3_bucket)
    print(f"Created S3 bucket: {s3_bucket}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"S3 bucket {s3_bucket} already exists")
except Exception as e:
    print(f"Error creating bucket: {e}")

## Store assets in S3



In [ ]:
import os

# Upload sample assets to S3 bucket
sample_assets_dir = 'sample_assets'

for root, dirs, files in os.walk(sample_assets_dir):
    for file in files:
        local_path = os.path.join(root, file)
        s3_key = local_path.replace('\\', '/')
        
        try:
            s3.upload_file(local_path, s3_bucket, s3_key)
            print(f"Uploaded: {local_path} -> s3://{s3_bucket}/{s3_key}")
        except Exception as e:
            print(f"Error uploading {local_path}: {e}")

print("Sample assets upload complete!")

In [ ]:
region_name = 'us-east-1'
os.environ['AWS_DEFAULT_REGION'] = region_name

: 

In [ ]:
# Store the bucket name for use in other notebooks
%store s3_bucket
%store region_name

## synchronous Embedding - Text

In [ ]:
from utils.utils import pretty_format
import nova_embeddings

request_body = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": 256,
        "text": {"truncationMode": "END", "value": "Hello, world!"},
    },
}

body, metadata = nova_embeddings.generate_embedding_sync(request_body)

print("Request ID:", metadata.get("RequestId"))
print(pretty_format(body))

### Synchronous Embedding - S3 Text File

This example shows how to use a text file stored in S3 as the input for a text embedding. **Be sure to edit the `S3_TEXT_FILE` value before running this cell.**

In [ ]:
from utils.utils import pretty_format
import nova_embeddings

# Update this constant to reference a text file in your S3 bucket.
S3_TEXT_FILE = f"s3://{s3_bucket}/sample_assets/text/Flatland-Edwin_Abbott-excerpt.txt"

request_body = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": 256,
        "text": {
            "truncationMode": "END",
            "source": {"s3Location": {"uri": S3_TEXT_FILE}},
        },
    },
}

body, metadata = nova_embeddings.generate_embedding_sync(request_body)

print("Request ID:", metadata.get("RequestId"))
print(pretty_format(body))

In [ ]:
import json

from utils.utils import load_file_as_base64
import nova_embeddings

request_body = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": 256,
        "video": {
            "format": "mp4",
            "embeddingMode": "AUDIO_VIDEO_COMBINED",
            "source": {
                "bytes": load_file_as_base64("sample_assets/videos/the-sea.mp4")
            },
        },
    },
}

body, metadata = nova_embeddings.generate_embedding_sync(request_body)

print("Request ID:", metadata.get("RequestId"))
print(json.dumps(body, indent=2))

Synchronous Emedding - S3 Video File
This next example creates an embedding for a video file stored in S3.



In [ ]:
import json

import nova_embeddings

# Update this constant to point to a video file in S3.
S3_VIDEO_FILE = f"s3://{s3_bucket}/sample_assets/videos/the-sea.mp4"

request_body = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": 3072,
        "video": {
            "format": "mp4",
            "embeddingMode": "AUDIO_VIDEO_COMBINED",
            "source": {
                "s3Location": {
                    "uri": S3_VIDEO_FILE,
                }
            },
        },
    },
}

body, metadata = nova_embeddings.generate_embedding_sync(request_body)

print("Request ID:", metadata.get("RequestId"))
print(json.dumps(body, indent=2))

### Asynchronous Embedding - Inline Video Data
Use the StartAsyncInvoke function to generate an embedding asynchronously. (See the generate_embedding_async() function definition in the "nova_embeddings.py" file for full details.) The results will be stored to an S3 bucket you specify. When generating an async embedding for video files, you can specify the segmentationConfig.durationSeconds value to dictate how longer videos will be broken into segments. The default is 5 seconds.

Calls to StartAsyncInvoke will return an invocationArn that can be used to check the status of the async job using the GetAsyncInvoke function.

In [ ]:
from utils.utils import load_file_as_base64
import nova_embeddings

# Update this constant to reference your own S3 bucket.
S3_DESTINATION_BUCKET = s3_bucket

request_body = {
    "taskType": "SEGMENTED_EMBEDDING",
    "segmentedEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": 256,
        "video": {
            "format": "mp4",
            "embeddingMode": "AUDIO_VIDEO_COMBINED",
            "source": {
                "bytes": load_file_as_base64("sample_assets/videos/the-sea.mp4")
            },
            "segmentationConfig": {"durationSeconds": 5},
        },
    },
}

invocation_arn, metadata = nova_embeddings.generate_embedding_async(
    request_body, S3_DESTINATION_BUCKET
)

print("Request ID:", metadata.get("RequestId"))
print("Invocation ARN:", invocation_arn)

# Document Embedding

In [ ]:
# Restore variables from setup notebook
%store -r s3_bucket
print(f"Using S3 bucket: {s3_bucket}")
%store -r region_name
print(f"Using region: {region_name}")

In [ ]:
!pip install chromadb

In [ ]:
from os import path, listdir
from pdf_to_png import pdf_to_png
from utils.utils import load_file_as_base64
import nova_embeddings
import chromadb

# Update the constants below if desired.
PDF_PATH = "sample_assets/documents/InTheHotSeat-WA.pdf"
EMBEDDING_DIMENSION = 3072

output_dir = path.splitext(PDF_PATH)[0]

pdf_to_png(pdf_path=PDF_PATH, output_dir=output_dir, dpi=200)

# Initialize the vector data store.
chroma_client = chromadb.PersistentClient()
collection_name = "document_pages"
try:
    chroma_client.delete_collection(collection_name)
except:
    pass  # Collection might not exist
collection = chroma_client.create_collection(collection_name)

# Generate embeddings for each image and add them to the vector store.
page_files = sorted(listdir(output_dir))
for filename in page_files:
    file_path = path.join(output_dir, filename)

    request_body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingDimension": EMBEDDING_DIMENSION,
            "embeddingPurpose": "GENERIC_INDEX",
            "image": {
                "format": "png",
                "detailLevel": "DOCUMENT_IMAGE",
                "source": {"bytes": load_file_as_base64(file_path)},
            },
        },
    }

    # Generate the embedding.
    print(f"Generating embedding for: {file_path}")
    body, metadata = nova_embeddings.generate_embedding_sync(request_body)
    embedding = body.get("embeddings")[0].get("embedding")

    # Add the image to the vector store.
    collection.add(
        ids=[file_path], embeddings=[embedding], metadatas=[{"file_path": file_path}]
    )

    print("Done")

In [ ]:
from os import path, listdir
from pdf_to_png import pdf_to_png
from utils.utils import load_file_as_base64
import nova_embeddings
import chromadb
from IPython.display import Image, display

# Updaate the constants below if desired.

QUERY_TEXT = "What are the signs of heat stroke?"
# QUERY_TEXT = "What is a heat dome?"
# QUERY_TEXT = "image of an ambulance"

EMBEDDING_DIMENSION = 3072

result_body, _ = nova_embeddings.generate_embedding_sync(
    {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": "DOCUMENT_RETRIEVAL",
            "embeddingDimension": EMBEDDING_DIMENSION,
            "text": {"truncationMode": "END", "value": QUERY_TEXT},
        },
    }
)

print("Generating query embedding...")
query_embedding = nova_embeddings.extract_embedding(result_body)

# Query the vector store.
collection = chromadb.PersistentClient().get_collection("document_pages")

# Query the collection for the top N most similar pages
retrieval_count = 3
results = collection.query(
    query_embeddings=[query_embedding], n_results=retrieval_count
)

# Display the results
print(f"Most relevant {retrieval_count} pages:")
for i, (doc_id, distance) in enumerate(zip(results["ids"][0], results["distances"][0])):
    print(f"Result {i+1}: {doc_id} (distance: {distance:.4f})")

print("\nMost relevant page image:")
first_image_path = results["ids"][0][0]
display(Image(first_image_path))

# Creating Text Query Embeddings

If you are creating an embedding to be used as a search query, using one of following query templates will improve ranking results.

**When comparing to text (e.g. in a text to text retrieval use-case):**
> Instruction: Given a query, retrieve passages that are relevant to the query:\nQuery: *{yourText}*

**When comparing to documents (e.g. in a text to document retrieval use-case):**
> Instruction: Retrieve a document that answers the following query:\nQuery: *{yourText}*

**When comparing to images, videos, or documents (e.g. in a text to multimodal and multi-file retrieval use-case):**
> Instruction: Find an image, video or document that match with the following description:\nQuery: *{yourText}*

The example below will let you explore the difference between using a non-optimized query embedding, and one that has been created using the optimized approach above.

Run the cell below to set up a helper function we'll use later.

In [ ]:
# Restore variables from setup notebook
%store -r s3_bucket
print(f"Using S3 bucket: {s3_bucket}")
%store -r region_name
print(f"Using region: {region_name}")

In [ ]:
from utils.utils import cosine_sim
import nova_embeddings


def sort_by_similarity(indexed_items, query_embedding):
    """Sort items by cosine similarity to a query embedding.

    Args:
        indexed_items (list): List of dictionaries, each containing an "embedding" key.
        query_embedding: The embedding vector to compare against.

    Returns:
        list: Sorted list of dictionaries with "similarity" and "item" keys,
              ordered by similarity score in descending order.
    """
    sorted_items = []
    for item in indexed_items:
        scored_item = {
            "similarity": cosine_sim(item["embedding"], query_embedding),
            "item": item,
        }
        sorted_items.append(scored_item)

    sorted_items.sort(key=lambda x: x["similarity"], reverse=True)
    return sorted_items

In [ ]:
embedding_dimension = 3072

In [ ]:
example_passages = [
    "The Science of Laughter: Why Giggles Might Be Humanity's Superpower",
    "Satellites, Selfies, and Space Junk: How Orbit Is Getting Crowded",
    "DIY DNA: How a Cup of Coffee Can Unlock Genetic Mysteries",
    "Rocket Science Isn't Hard? Meet the Everyday Physics Behind Liftoff",
    "Ant Cities vs. Human Cities: Who Builds Better?",
    "Why Mars Wants You: The Surprising Skills Future Space Colonists Will Need",
    "Volcanoes Under Ice: The Hidden Heat Shaping Our Planet",
    "The Secret Life of Bananas: How Your Fruit Bowl Explains Evolution",
    "The Cosmic Ocean: What Jellyfish Teach Us About Traveling the Stars",
    "Time Travel for Beginners: Why Your Microwave Is Already Bending Physics",
]

passages_with_embeddings = []

for index, passage in enumerate(example_passages):
    print(
        f"\rGenerating embedding {index + 1} of {len(example_passages)}",
        end="",
        flush=True,
    )

    indexing_embedding_params = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": "GENERIC_INDEX",
            "embeddingDimension": embedding_dimension,
            "text": {"truncationMode": "END", "value": passage},
        },
    }
    result_body, _ = nova_embeddings.generate_embedding_sync(indexing_embedding_params)
    embedding = nova_embeddings.extract_embedding(result_body)
    passages_with_embeddings.append(
        {"index": index, "text": passage, "embedding": embedding}
    )

In [ ]:
result_body, _ = nova_embeddings.generate_embedding_sync(
    {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": "TEXT_RETRIEVAL",
            "text": {"truncationMode": "END", "value": query_text},
        },
    }
)

query_embedding = nova_embeddings.extract_embedding(result_body)

# Create a list of passages sorted by cosine similarity.
sorted_passages = sort_by_similarity(passages_with_embeddings, query_embedding)

# Print the sorted list.
for item in sorted_passages:
    print(f"Similarity: {item['similarity']:.6f}, Text: {item['item']['text']}")